# Crivo: inspeção do intérprete contextual próprio (2,95M)

Este notebook carrega os pesos **já treinados** do experimento e permite inspecionar propostas e fontes. **O candidato não passou no critério do piloto e não está aprovado para o chat.** Acertou 342/400 contratos dentro de uma gramática limitada, mas apenas 24/56 requisitos e 0/4 na transferência adicional de preços. Não produz conversa livre.

A execução padrão não repete treinamento. Só usa arquitetura, tokenizer e pesos próprios do CRIVO. Torch, NumPy e tokenizers são infraestrutura. Não acessa modelos externos ou conversas privadas.


In [ ]:
from pathlib import Path
import subprocess, sys, json
REPO = Path('/content/CRIVO-contextual')
BRANCH = 'codex/interpretador-contextual-20261008'
if not REPO.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                           'https://github.com/HROSONE/CRIVO.git', str(REPO)])
else:
    branch = subprocess.check_output(['git','branch','--show-current'],cwd=REPO,text=True).strip()
    if branch != BRANCH:
        raise RuntimeError('A pasta pertence a outra branch; use uma pasta nova.')
REVISAO = subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
EXP = REPO / 'experimentos/interpretador_contextual_20261008'
print('Revisão do experimento:', REVISAO)


In [ ]:
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'torch>=2.2,<3', 'numpy>=1.24,<3', 'tokenizers>=0.20,<1'])


## Resultados conservados

Os acertos incluem operação, argumentos, fonte, referente e escopo. A cópia limitada usa gramática determinística junto aos pesos. Estes resultados não são uma nota de conversa livre ou avaliação independente.


In [ ]:
r = json.loads((EXP/'avaliacao/resultado.json').read_text())
for k in ['candidato_argumentos','candidato_limitado','atingiu_utilidade_piloto_limitado','aprovado_para_chat']:
    print(k, json.dumps(r[k],ensure_ascii=False))


## Inspecionar uma conversa

Edite a lista abaixo. Os trechos são somente entrada de inferência: não são acrescentados ao treino nem salvos no Drive. Históricos acima de 256 tokens são recusados. `executavel` confere a operação, mas o modelo ainda pode selecionar fatos errados.


In [ ]:
FALAS = [
    'A opção A custa 23 reais. A opção B custa 35 reais. Qual opção custa menos?',
    'Corrijo o preço da opção A: agora custa 41 reais. Compare os custos das opções.'
]
entrada = Path('/content/crivo-entrada-contextual.json')
entrada.write_text(json.dumps(FALAS,ensure_ascii=False))
subprocess.check_call([sys.executable,str(EXP/'inferir.py'),
    '--pesos',str(EXP/'pesos_contextual/pesos.pt'),
    '--tokenizer',str(REPO/'artefatos/linguagem_profunda/tokenizer.json'),
    '--entrada',str(entrada),'--raiz',str(REPO)])


## Reprodução opcional do treino

A rodada abaixo reproduz a receita publicada; não se espera que resolva as falhas conhecidas. O padrão é desligado para evitar repetir treino desnecessariamente. Na medição local, cada braço levou aproximadamente 11–12 minutos em CPU; o Colab pode ter desempenho diferente. Este script usa CPU.

Se ativar, os resultados ficam em uma pasta nova no seu Drive. Uma sessão interrompida antes do checkpoint final não tem retomada automática de Adam/RNG neste piloto. Não apague rodadas anteriores para recomeçar. A avaliação publicada já é conhecida; não representa um novo teste cego.


In [ ]:
TREINAR_NOVAMENTE = False
if TREINAR_NOVAMENTE:
    from google.colab import drive
    from datetime import datetime, timezone
    import shutil, os
    drive.mount('/content/drive')
    rodada = Path('/content/drive/MyDrive/CRIVO/interpretador-contextual') / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    rodada.mkdir(parents=True,exist_ok=False)
    shutil.copytree(EXP/'dados',rodada/'dados')
    (rodada/'origem.json').write_text(json.dumps({'revisao':REVISAO,'aprovado':False}))
    ambiente = dict(os.environ,OPENBLAS_NUM_THREADS='1',OMP_NUM_THREADS='1')
    for braco in ['controle','contextual']:
        subprocess.check_call([sys.executable,str(EXP/'treinar.py'),braco,str(rodada/braco),
            '--raiz',str(REPO),'--dados',str(rodada/'dados')],env=ambiente)
    subprocess.check_call([sys.executable,str(EXP/'avaliar.py'),
        '--raiz',str(REPO),'--experimento',str(rodada)],env=ambiente)
    print('Rodada conservada:',rodada)
else:
    print('Treino desligado. Os pesos publicados já podem ser inspecionados acima.')


## Próximo avanço necessário

Variar relações, nomes de opções, ordem e formas das declarações; testar normalização de entidades com vínculo à fonte. Preparar outro painel antes de novo treino. A geração de diálogo aberto precisa de um currículo de redação separado e leitura manual. Nenhum resultado deste notebook ativa pesos no chat.
